# NBA Player Over/Under Prediction — ML Model Building

Predicts whether a player's PTS + REB + AST will finish **over** or **under**
a line set from their prior 10-game average.

**v2 — rebuilt to remove target leakage.**
The first version fed same-game box-score columns (`FGM`, `FG3M`, `FTM`,
`OREB`, `DREB`, `NBA_FANTASY_PTS`) into the model while the target was derived
from that same game's PTS + REB + AST. Since points are recoverable as
`2*FGM + FG3M + FTM` and rebounds as `OREB + DREB`, the models were
reconstructing the target rather than forecasting it — which is why every model
landed near 0.84 AUC while an honest statistical baseline scored 0.523.

All features now come from **prior games only**, built by `build_features.py`.
The train/test split is **chronological**, not random.

In [ ]:
import os
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import cross_val_score
from sklearn.metrics import roc_auc_score, brier_score_loss

warnings.filterwarnings('ignore')

import build_features as bf

# Everything below uses paths relative to the project root, so fail loudly
# here rather than with a confusing empty-glob error three cells down.
PROJECT = Path.cwd()
if not (PROJECT / 'build_features.py').exists():
    raise RuntimeError(
        f'Notebook is running from {PROJECT}, which is not the project root.\n'
        'Open the player-prediction FOLDER in VS Code, not the notebook file.'
    )

DATA = PROJECT / 'data' / 'raw'
MODELS = PROJECT / 'models'
MODELS.mkdir(exist_ok=True)

print(f'Project root: {PROJECT}')
print('All libraries loaded successfully.')

## Step 1: Load Game Logs & Build Lagged Features

In [ ]:
files = sorted(DATA.glob('nba_player_gamelogs_*.csv'))
if not files:
    raise FileNotFoundError(f'No game logs in {DATA}')

print(f'Found {len(files)} files')
raw = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
print(f'Raw shape: {raw.shape}')

# build() handles cleaning, rolling windows, context and opponent features,
# and the target. Every feature is shifted so it contains prior games only.
df, feats = bf.build(raw)
print(f'Feature-engineered shape: {df.shape}')

## Step 2: Leakage Check

The guard that makes this version different from v1. It raises if any
same-game column ends up in the feature list. Leakage is easy to reintroduce
and silent when it happens, so this runs before every training run.

In [ ]:
bf.leakage_check(feats)

print(f'\nFeatures: {len(feats)}')
print(f'Target distribution: {df["TARGET"].mean():.2%} OVER')
print(f'\nSample feature names:')
for f in feats[:12]:
    print(f'  {f}')
print(f'  ... and {len(feats) - 12} more')

## Step 3: Chronological Train/Test Split

v1 used `train_test_split(..., random_state=42)`, which trains on March games
and tests on January ones — a situation you never face live. Time-ordered data
gets a time-ordered split.

In [ ]:
train, test = bf.chronological_split(df, test_frac=0.2)

X_train, Y_train = train[feats], train['TARGET']
X_test,  Y_test  = test[feats],  test['TARGET']

print(f'\nTrain OVER%: {Y_train.mean():.2%}')
print(f'Test  OVER%: {Y_test.mean():.2%}')

## Step 4: Scale Features

In [ ]:
# Fit on train only — fitting on the full set would leak test-set
# distribution into the scaler.
scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f'Scaled: {X_train_s.shape[0]:,} train / {X_test_s.shape[0]:,} test rows')

## Step 5: Feature Selection for Model 2

ANOVA F-test picks the top 10 features for the reduced-feature model.

Caveat worth knowing: the F-test scores each feature independently, and the
rolling averages are heavily correlated with each other (`FGM_avg3`,
`FGM_avg5`, and `FGM_avg10` are nearly the same column). So the selected set
will likely contain near-duplicates. Recursive feature elimination or L1
regularization would handle that better — this is kept because the original
assignment specified `SelectKBest`.

In [ ]:
selector = SelectKBest(score_func=f_classif, k=10).fit(X_train, Y_train)

feature_scores = pd.DataFrame({
    'Feature': feats,
    'F_Score': selector.scores_,
    'P_Value': selector.pvalues_,
}).sort_values('F_Score', ascending=False)

print('Top 15 features by F-score:')
print(feature_scores.head(15).to_string(index=False))

selected_features = [f for f, keep in zip(feats, selector.get_support()) if keep]
print(f'\nSelected for M2: {selected_features}')

X_train_reduced = X_train[selected_features]
X_test_reduced  = X_test[selected_features]

## Step 6: Model 1 — Random Forest

`min_samples_leaf=20` matters here. With 73 features and no leaked signal to
paper over it, an unconstrained forest memorizes noise.

In [ ]:
model_1 = RandomForestClassifier(
    n_estimators=300,
    min_samples_leaf=20,
    random_state=42,
    n_jobs=-1,
)
model_1.fit(X_train, Y_train)

proba_m1 = model_1.predict_proba(X_test)[:, 1]
result_1 = model_1.score(X_test, Y_test)
auc_m1   = roc_auc_score(Y_test, proba_m1)

print(f'M1 Random Forest — Acc {result_1:.4f}  AUC {auc_m1:.4f}  '
      f'Brier {brier_score_loss(Y_test, proba_m1):.4f}')

m1_bundle = {'model': model_1, 'features': feats, 'version': 'v2-lagged'}
pickle.dump(m1_bundle, open(MODELS / 'finalized_model_M1.sav', 'wb'))
print(f'Saved: {MODELS / "finalized_model_M1.sav"}')

# Top drivers — worth a look now that the features are honest.
imp = pd.Series(model_1.feature_importances_, index=feats).sort_values(ascending=False)
print('\nTop 10 features by importance:')
print(imp.head(10).to_string())

## Step 7: Model 2 — Logistic Regression (Reduced Feature Set)

In [ ]:
scaler_m2 = StandardScaler().fit(X_train_reduced)
model_2 = LogisticRegression(max_iter=2000, random_state=42, C=1.0)
model_2.fit(scaler_m2.transform(X_train_reduced), Y_train)

proba_m2 = model_2.predict_proba(scaler_m2.transform(X_test_reduced))[:, 1]
result_2 = model_2.score(scaler_m2.transform(X_test_reduced), Y_test)
auc_m2   = roc_auc_score(Y_test, proba_m2)

print(f'M2 Logistic Regression — Acc {result_2:.4f}  AUC {auc_m2:.4f}  '
      f'Brier {brier_score_loss(Y_test, proba_m2):.4f}')

m2_bundle = {'scaler': scaler_m2, 'model': model_2,
             'features': selected_features, 'version': 'v2-lagged'}
pickle.dump(m2_bundle, open(MODELS / 'finalized_model_M2.sav', 'wb'))
print(f'Saved: {MODELS / "finalized_model_M2.sav"}')

## Step 8: Model 3 — KNN

KNN scales badly: every prediction compares against all 261k training rows.
The elbow search runs on a subsample, then the final model trains on
everything. Without the subsample this cell takes hours.

In [ ]:
rng = np.random.default_rng(42)
sub = rng.choice(len(X_train_s), size=min(30_000, len(X_train_s)), replace=False)
X_sub, Y_sub = X_train_s[sub], Y_train.iloc[sub]

k_range = range(3, 31, 4)
cv_scores = []
for k in k_range:
    knn = KNeighborsClassifier(n_neighbors=k, metric='euclidean',
                               weights='distance', n_jobs=-1)
    cv = cross_val_score(knn, X_sub, Y_sub, cv=3, scoring='roc_auc', n_jobs=-1)
    cv_scores.append(cv.mean())
    print(f'  K={k:2}  CV AUC {cv.mean():.4f}')

best_k = list(k_range)[int(np.argmax(cv_scores))]

plt.figure(figsize=(9, 4))
plt.plot(list(k_range), cv_scores, marker='o', color='steelblue')
plt.axvline(best_k, color='tomato', linestyle='--', label=f'best K={best_k}')
plt.xlabel('K (number of neighbors)')
plt.ylabel('CV AUC (3-fold, 30k subsample)')
plt.title('KNN — K Selection')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'\nBest K: {best_k}  (CV AUC {max(cv_scores):.4f})')

### Train final KNN

In [ ]:
model_3 = KNeighborsClassifier(n_neighbors=best_k, metric='euclidean',
                               weights='distance', n_jobs=-1)
model_3.fit(X_train_s, Y_train)

proba_m3 = model_3.predict_proba(X_test_s)[:, 1]
result_3 = model_3.score(X_test_s, Y_test)
auc_m3   = roc_auc_score(Y_test, proba_m3)

print(f'M3 KNN K={best_k} — Acc {result_3:.4f}  AUC {auc_m3:.4f}  '
      f'Brier {brier_score_loss(Y_test, proba_m3):.4f}')

m3_bundle = {'scaler': scaler, 'model': model_3, 'features': feats,
             'k': best_k, 'version': 'v2-lagged'}
pickle.dump(m3_bundle, open(MODELS / 'finalized_model_M3.sav', 'wb'))
print(f'Saved: {MODELS / "finalized_model_M3.sav"}')

## Step 9: Compare All Three Models

In [ ]:
from sklearn.metrics import roc_curve

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
for proba, label in [
    (proba_m1, f'Random Forest (AUC={auc_m1:.3f})'),
    (proba_m2, f'Logistic Reg. (AUC={auc_m2:.3f})'),
    (proba_m3, f'KNN K={best_k} (AUC={auc_m3:.3f})'),
]:
    fpr, tpr, _ = roc_curve(Y_test, proba)
    ax.plot(fpr, tpr, label=label, linewidth=2)
ax.plot([0, 1], [0, 1], 'k--', alpha=0.4, label='Random baseline')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — Lagged Features')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

ax2 = axes[1]
names  = ['RF (M1)', 'LR (M2)', f'KNN K={best_k} (M3)']
aucs   = [auc_m1, auc_m2, auc_m3]
bars = ax2.bar(names, aucs, color=['steelblue', 'tomato', 'seagreen'],
               alpha=0.85, edgecolor='white')
for bar, v in zip(bars, aucs):
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.002,
             f'{v:.4f}', ha='center', va='bottom', fontweight='bold')
ax2.axhline(0.5, color='k', linestyle='--', alpha=0.4)
ax2.set_ylim(0.45, max(aucs) + 0.05)
ax2.set_ylabel('Test AUC')
ax2.set_title('AUC Comparison (0.5 = random)')
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## Step 10: Ensemble

In v1 the ensemble beat the best single model by 0.0013 AUC on 4,877 test
rows — noise, not an improvement. Check whether that still holds here before
claiming the ensemble is better than Random Forest alone.

In [ ]:
proba_ens = (proba_m1 + proba_m2 + proba_m3) / 3
auc_ens = roc_auc_score(Y_test, proba_ens)
result_ens = ((proba_ens > 0.5).astype(int) == Y_test).mean()

print(f'Ensemble — Acc {result_ens:.4f}  AUC {auc_ens:.4f}  '
      f'Brier {brier_score_loss(Y_test, proba_ens):.4f}')

best_single = max(auc_m1, auc_m2, auc_m3)
delta = auc_ens - best_single
print(f'\nvs best single model: {delta:+.4f} AUC')
if abs(delta) < 0.005:
    print('Within noise — report as comparable, not better.')

## Step 11: Calibration

AUC says the model ranks OVERs above UNDERs. It says nothing about whether
"62% confident" means 62% of the time. For a prop model that's the number
that matters, since a predicted probability gets compared against a line's
implied probability to size a position. A well-ranked but miscalibrated model
is useless for that.

In [ ]:
from sklearn.calibration import calibration_curve

plt.figure(figsize=(7, 6))
for proba, label in [(proba_m1, 'RF'), (proba_m2, 'LR'),
                     (proba_m3, f'KNN K={best_k}'), (proba_ens, 'Ensemble')]:
    frac_pos, mean_pred = calibration_curve(Y_test, proba, n_bins=10)
    plt.plot(mean_pred, frac_pos, marker='o', label=label)

plt.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Perfectly calibrated')
plt.xlabel('Mean predicted probability')
plt.ylabel('Observed fraction positive')
plt.title('Calibration — Reliability Diagram')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Summary

In [ ]:
print('=' * 62)
print('   NBA OVER/UNDER — v2 (LAGGED FEATURES, CHRONOLOGICAL SPLIT)')
print('=' * 62)
print(f'  Rows after feature engineering: {len(df):,}')
print(f'  Train: {len(X_train):,}   Test: {len(X_test):,}')
print(f'  Features: {len(feats)} (all from prior games)')
print(f'  Test OVER rate: {Y_test.mean():.2%}')
print()
print(f'  {"Model":<26}{"Acc":>9}{"AUC":>9}{"Brier":>9}')
print('  ' + '-' * 53)
for name, acc, auc, proba in [
    ('Random Forest (M1)', result_1, auc_m1, proba_m1),
    ('Logistic Regression (M2)', result_2, auc_m2, proba_m2),
    (f'KNN K={best_k} (M3)', result_3, auc_m3, proba_m3),
    ('Ensemble', result_ens, auc_ens, proba_ens),
]:
    print(f'  {name:<26}{acc:>9.4f}{auc:>9.4f}'
          f'{brier_score_loss(Y_test, proba):>9.4f}')
print('=' * 62)
print()
print('  v1 (same-game features, random split): 0.8425 AUC ensemble')
print('  The gap between that and the numbers above is the leakage.')
print()
print('  Target: PTS + REB + AST vs prior 10-game average')
print(f'  Models saved to: {MODELS}')